**Joins — inner, left, right, full outer**

| Join Type | What it does | Rows Returned |
|-----------|--------------|---------------|
| `inner` | Returns only the rows that have matching keys in both DataFrames | Matching rows only |
| `left` (`left_outer`) | Returns all rows from the left DataFrame and matching rows from the right. Non-matching right values become `NULL`. | All left rows |
| `right` (`right_outer`) | Returns all rows from the right DataFrame and matching rows from the left. Non-matching left values become `NULL`. | All right rows |
| `full` (`full_outer`) | Returns all rows from both DataFrames. Non-matching values on either side become `NULL`. | All rows from both DataFrames |
| `left_semi` | Returns only the rows from the left DataFrame that have a match in the right DataFrame. Columns from the right DataFrame are **not** included. | Matching left rows only |
| `left_anti` | Returns only the rows from the left DataFrame that **do not** have a match in the right DataFrame. | Non-matching left rows only |
| `cross` | Returns the Cartesian product of both DataFrames (every left row paired with every right row). | `Left Rows × Right Rows` |

In [28]:
# Create Spark session
# Hadoop AWS connector allows Spark to communicate with Amazon S3
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Day-11")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)


In [29]:
customers_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/customers.csv')
orders_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/orders.csv')
products_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/products.csv')

**Task 1**

Join orders.csv with customers.csv using an inner join on customer_id. Select order_id, first_name, last_name, unit_price, and region. Show the first 5 rows. What is the row count?

In [30]:
joined_df=orders_df.join(customers_df,on='customer_id',how='inner')
joined_df.select('order_id','first_name','last_name',
'unit_price','region').show(5)
print(f"row count:{joined_df.count()}")

+--------+----------+---------+----------+-------+
|order_id|first_name|last_name|unit_price| region|
+--------+----------+---------+----------+-------+
|   O0001|     James| Anderson|   1299.99|   East|
|   O0002|     Maria|   Garcia|    449.99|   West|
|   O0003|    Robert|  Johnson|    349.99|Midwest|
|   O0004|     Linda| Martinez|     89.99|  South|
|   O0005|   Michael|    Brown|     29.99|   West|
+--------+----------+---------+----------+-------+
only showing top 5 rows


row count:100


**Task 2**

Join orders.csv with customers.csv using a left join. How many orders have no matching customer? Filter for those rows and show them.

In [31]:
left_joined_df = orders_df.join(customers_df, on="customer_id", how="left")

print(f"Orders : {orders_df.count()}")
print(f"After join : {left_joined_df.count()}")
unmatched = left_joined_df.filter(F.col("first_name").isNull())
print(f"Orders with no customer record: {unmatched.count()}")


Orders : 100


After join : 100


Orders with no customer record: 0


**Task 3**

Join orders.csv with products.csv using an inner join on product_id. Both DataFrames have a unit_price column. Handle the duplicate by selecting orders_df["unit_price"].alias("order_price") and products_df["unit_price"].alias("list_price"). Show the result.

In [32]:
from pyspark.sql import functions as F

joined_df = (
    orders_df.alias("o")
    .join(
        products_df.alias("p"),
        on="product_id",
        how="inner"
    )
    .select(
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.col("o.product_id"),
        F.col("o.order_date"),
        F.col("o.quantity"),
        F.col("o.unit_price").alias("order_price"),
        F.col("p.unit_price").alias("list_price"),
        F.col("p.product_name"),
        F.col("p.category")
    )
)

joined_df.show(5,truncate=False)

+--------+-----------+----------+----------+--------+-----------+----------+-------------------+-----------+
|order_id|customer_id|product_id|order_date|quantity|order_price|list_price|product_name       |category   |
+--------+-----------+----------+----------+--------+-----------+----------+-------------------+-----------+
|O0001   |C001       |P001      |2023-01-05|2       |1299.99    |1299.99   |Laptop Pro 15      |Electronics|
|O0002   |C002       |P005      |2023-01-07|1       |449.99     |449.99    |Monitor 27inch 4K  |Electronics|
|O0003   |C003       |P003      |2023-01-10|4       |349.99     |349.99    |Office Chair Deluxe|Furniture  |
|O0004   |C004       |P006      |2023-01-12|2       |89.99      |89.99     |Mechanical Keyboard|Electronics|
|O0005   |C005       |P002      |2023-01-15|3       |29.99      |29.99     |Wireless Mouse     |Electronics|
+--------+-----------+----------+----------+--------+-----------+----------+-------------------+-----------+
only showing top 5 

**Task 4**

Perform a full outer join between customers.csv and orders.csv. Find customers who have never placed an order. How many are there?

In [ ]:
full_join_df = customers_df.join(
    orders_df,
    on="customer_id",
    how="full"
)

customers_without_orders = full_join_df.filter(
    F.col("order_id").isNull()
)

customers_without_orders.show(truncate=False)

print("Customers who never placed an order:", customers_without_orders.count())

+-----------+----------+---------+-----+----+-----+-------+-----------+-------+--------+----------+----------+--------+----------+------------+------+--------------+------+
|customer_id|first_name|last_name|email|city|state|country|signup_date|segment|order_id|product_id|order_date|quantity|unit_price|discount_pct|status|payment_method|region|
+-----------+----------+---------+-----+----+-----+-------+-----------+-------+--------+----------+----------+--------+----------+------------+------+--------------+------+
+-----------+----------+---------+-----+----+-----+-------+-----------+-------+--------+----------+----------+--------+----------+------------+------+--------------+------+



Customers who never placed an order: 0
